# 노션 트러블슈팅 검색기


**프로젝트 개요**

기존 노션 검색의 한계인 제목 기반 검색을 벗어나, 문서 내용까지 의미적으로 검색할 수 있는 시스템을 구현했습니다.
```

노션 DB Export (Markdown)
        ↓
문서 청킹 (RecursiveCharacterTextSplitter)
        ↓
임베딩 변환 (OpenAI Embeddings)
        ↓
벡터 DB 저장 (Chroma)
        ↓
의미 기반 검색
        ↓
관련 문서 반환
```


In [2]:
!pip3 install -qU langchain-chroma
!pip3 install -qU langchain-openai
!pip3 install -qU langchain-text-splitters
!pip3 install -qU gradio python-dotenv
!pip3 install -qU "huggingface_hub==0.34.4"


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


### API 키 설정

In [3]:
from pathlib import Path
from datetime import datetime
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from dotenv import load_dotenv
import gradio as gr
import os
import json

load_dotenv()
print("OPENAI_API_KEY loaded:", os.getenv("OPENAI_API_KEY") is not None)

# API 키 잘 로드됐는지 확인
print(os.environ.get("OPENAI_API_KEY")[:5] + "*****")

embeddings = OpenAIEmbeddings()
print("✅ 임베딩 모델 준비 완료!")


/Users/eunchae/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/eunchae/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OPENAI_API_KEY loaded: True
sk-pr*****
✅ 임베딩 모델 준비 완료!


### 노션에서 Export 한 zip 파일 압축 해제

In [4]:
import zipfile
import os

# 압축 풀기
# 이미 압축 풀렸으면 스킵!
if not os.path.exists("./notion_docs"):
    with zipfile.ZipFile("notion_export.zip", "r") as zip_ref:
        zip_ref.extractall("./notion_docs")
    print("✅ 압축 해제 완료!")
else:
    print("✅ 이미 압축 해제됨, 스킵!")

# 파일 목록 확인
#for root, dirs, files in os.walk("./notion_docs"):
#    for file in files:
#        print(file)


✅ 이미 압축 해제됨, 스킵!


In [5]:
# ============================================
# md 파일 경로 수집
# ============================================
import glob

md_paths = glob.glob("./notion_docs/**/*.md", recursive=True)
md_paths = [p for p in md_paths if not p.endswith("SUMMARY.md")]
print(f"📄 총 {len(md_paths)}개 md 파일 발견!")

📄 총 56개 md 파일 발견!


In [6]:
# ============================================
# 문서 읽기
# ============================================
docs = [] # LangChain Document 객체 리스트. 벡터 DB 저장용
original_docs = {} # 내용 전체
doc_meta = {}

for full_path in md_paths:
    with open(full_path, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            continue

        path_obj = Path(full_path)
        title = path_obj.stem
        stat = path_obj.stat()
        created_ts = getattr(stat, "st_birthtime", stat.st_mtime)
        created_at = datetime.fromtimestamp(created_ts).strftime("%Y-%m-%d")

        original_docs[title] = content
        doc_meta[title] = {
            "created_at": created_at,
            "created_ts": created_ts,
            "full_path": full_path,
        }
        docs.append(Document(
            page_content=content,
            metadata={
                "source": f"{title}.md",
                "full_path": full_path,
                "created_at": created_at,
            },
        ))

print(f"✅ 원본 문서 {len(original_docs)}개 준비 완료")
print("\n파일 목록:")
for path in md_paths[:5]:
    print(" -", path)

✅ 원본 문서 56개 준비 완료

파일 목록:
 - ./notion_docs/troubleshooting/PR 서버에 적용하면서 문제 및 개선사항 정리 (feat dentrion_tablet_ap 19aaca7eedd180debcdcd3e399ada17e.md
 - ./notion_docs/troubleshooting/@[트러블슈팅]MySQL 버전 문제 (ONLY_FULL_GROUP_BY 모드) 292aca7eedd180ef858bc549f96f0b7f.md
 - ./notion_docs/troubleshooting/네트워크 699e097547da496d9b5e57952d80156d.md
 - ./notion_docs/troubleshooting/springdoc - array 타입이 스웨거에서 이상하게 표시 198aca7eedd18065b211d0967839d244.md
 - ./notion_docs/troubleshooting/db 이관 후 테이블 대소문자 구분 안됨 ae65f04015184caaa30013127611c1c6.md


### 벡터 DB 설정

In [7]:

import shutil

#DB 업데이트 여부 설정 
FORCE_UPDATE = False  # 처음엔 True로!

In [8]:

if not os.path.exists("./notion_chroma") or FORCE_UPDATE:
    # 기존 DB 삭제
    if os.path.exists("./notion_chroma"):
        shutil.rmtree("./notion_chroma")
        print("🗑️ 기존 벡터 DB 삭제!")

    #새로만들기
    chunks = text_splitter.split_documents(docs)
    print(f"✂️ 총 {len(chunks)}개 청크!")

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name="troubleshooting_db",
        persist_directory="./notion_chroma"
    )
    print(f"✅ 저장 완료! {vector_store._collection.count()}개")
else:
    # 기존 DB 로드
    vector_store = Chroma(
        collection_name="troubleshooting_db",
        embedding_function=embeddings,
        persist_directory="./notion_chroma"
    )
    print(f"✅ 기존 DB 로드! {vector_store._collection.count()}개")

    

✅ 기존 DB 로드! 662개


### 프롬프트

In [9]:
query_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

query_prompt = ChatPromptTemplate.from_template("""
너는 트러블슈팅 검색을 위한 질의 정규화기다.

목표:
- 사용자 질문에서 핵심 기술 키워드를 추출
- 한글/영문/대소문자가 다른 동일 개념을 같은 의미로 처리
- 검색에 유리하도록 대표 검색어와 확장 검색어를 생성

동의어 규칙 예시:
- jenkins = Jenkins = 젠킨스
- github = GitHub = 깃허브
- swagger = Swagger = 스웨거
- spring boot = Spring Boot = 스프링부트
- mysql = MySQL = 마이에스큐엘
- tls = TLS = ssl = SSL

지침:
- 같은 의미의 표현은 하나의 개념으로 묶기
- 원문 의미를 바꾸지 말기
- 검색에 불필요한 조사나 장식어는 제거 가능
- 출력은 반드시 JSON 형태로만 반환

출력 형식:
{{
  "normalized_query": "대표 검색어",
  "expanded_keywords": ["동의어1", "동의어2", "동의어3"],
  "intent": "사용자가 찾고 싶은 문제 상황 요약"
}}

사용자 질문:
{question}
""")

print("✅ 프롬프트 정의 완료!")




✅ 프롬프트 정의 완료!


### 쿼리 정규화 함수

In [10]:


query_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def normalize_query(question: str) -> dict:
    """사용자 질문을 정규화하여 검색어 확장"""
    response = query_llm.invoke(query_prompt.format(question=question))
    return json.loads(response.content)

def dedupe_by_title(documents: list) -> list:
    """제목 기준 중복 문서 제거"""
    seen = set()
    unique_docs = []
    for doc in documents:
        title = doc.metadata.get("source", "unknown").replace(".md", "")
        if title not in seen:
            seen.add(title)
            unique_docs.append(doc)
    return unique_docs

def search_documents(vector_store, question: str, k: int = 8) -> dict:
    """벡터 DB에서 관련 문서 검색"""
    query_info = normalize_query(question)
    queries = [query_info["normalized_query"]] + query_info["expanded_keywords"]

    results = []
    seen = set()
    for q in queries:
        docs = vector_store.similarity_search(q, k=k)
        for doc in docs:
            key = (doc.metadata.get("source", ""), doc.page_content[:200])
            if key not in seen:
                seen.add(key)
                results.append(doc)

    return {
        "query_info": query_info,
        "documents": dedupe_by_title(results)[:k],
    }

print("✅ 검색 함수 정의 완료!")

✅ 검색 함수 정의 완료!


### Gradio 함수

In [24]:
# ============================================
# Gradio 함수
# ============================================

def make_item(title, mode="all"):
    """문서 아이템 생성"""
    if mode == "all":
        subtitle = f"🗓️ {doc_meta[title]['created_at']}"
    else:
        snippet = original_docs.get(title, "").replace("\n", " ").strip()[:140]
        subtitle = f"📝 {snippet}..."
    
    return {
        "title": title,
        "mode": mode,
        "subtitle": subtitle  # ← 항상 subtitle 키 포함!
    }

def build_items(query: str) -> list:
    """검색어 유무에 따라 문서 목록 생성"""
    query = (query or "").strip()
    if not query:
        # 검색어 없으면 전체 문서 최신순
        titles = sorted(
            original_docs.keys(),  # ← docs → original_docs!
            key=lambda x: doc_meta[x]["created_ts"],
            reverse=True,
        )
        return [make_item(title, mode="all") for title in titles]

    # 검색어 있으면 검색 결과
    result = search_documents(vector_store, query, k=10)
    titles = [
        doc.metadata["source"].replace(".md", "")
        for doc in result["documents"]
    ]
    return [make_item(title, mode="search") for title in titles]
    
def search(query: str):
    """검색 실행"""
    items = build_items(query)
    titles = [item["title"] for item in items]
    return gr.update(choices=titles, value=None), "", "문서 제목을 클릭하면 전체 내용을 볼 수 있어요."

def show_full_doc(title: str):
    if not title:
        return "", "문서를 찾을 수 없어요"
    content = original_docs.get(title, "문서를 찾을 수 없어요")
    # return f"## 📄 {title}", content

    return f"## 📄 {title}\n\n---\n\n{content}"

def create_show_handler(title: str):
    """문서 클릭 핸들러 생성"""
    def _handler():
        return show_full_doc(title)
    return _handler

print("✅ Gradio 함수 정의 완료!")

✅ Gradio 함수 정의 완료!


### Gradio UI

In [12]:
# Gradio UI 셀 전에 실행!

# 1. 데이터 확인
print("=== 데이터 확인 ===")
print(f"original_docs: {len(original_docs)}개")
print(f"doc_meta: {len(doc_meta)}개")

# 2. build_items 결과 확인
items = build_items("")
print(f"\n=== build_items 결과 ===")
print(f"총 {len(items)}개")
print(f"샘플: {items[:2]}")

# 3. make_item 직접 테스트
title = list(original_docs.keys())[0]
item = make_item(title, mode="all")
print(f"\n=== make_item 결과 ===")
print(item)

=== 데이터 확인 ===
original_docs: 56개
doc_meta: 56개

=== build_items 결과 ===
총 56개
샘플: [{'title': 'tablet 모듈 신규 생성 시 bean 주입 에러 e14a4b3c072c4cf48f05f684b0b83a91', 'mode': 'all', 'subtitle': '🗓️ 2026-04-07'}, {'title': '덴트리온 거래명세서 temp 테이블 fbea6bf287944d039d469e55650e803a', 'mode': 'all', 'subtitle': '🗓️ 2026-04-07'}]

=== make_item 결과 ===
{'title': 'PR 서버에 적용하면서 문제 및 개선사항 정리 (feat dentrion_tablet_ap 19aaca7eedd180debcdcd3e399ada17e', 'mode': 'all', 'subtitle': '🗓️ 2026-04-07'}


In [25]:
# ============================================
# Gradio UI
# ============================================

custom_css = """
#doc-left-panel,
#doc-right-panel {
  height: 720px;
  overflow-y: auto;
}
#doc-left-panel {
  border-right: 1px solid #e5e7eb;
  padding-right: 12px;
}
#doc-right-panel {
  padding-left: 12px;
}
.doc-card {
  border-bottom: 1px solid #e5e7eb;
  padding: 10px 0;
}
.doc-title-btn button {
  justify-content: flex-start !important;
  text-align: left !important;
  font-size: 1.05rem !important;
  font-weight: 700 !important;
  border: none !important;
  background: transparent !important;
  box-shadow: none !important;
  padding: 0 !important;
  color: #1f2937 !important;
}
.doc-date p {
  font-size: 0.76rem !important;
  color: #6b7280 !important;
  margin-top: 4px !important;
  margin-bottom: 0 !important;
}
.doc-preview p {
  font-size: 0.88rem !important;
  color: #6b7280 !important;
  margin-top: 4px !important;
  margin-bottom: 0 !important;
  line-height: 1.5 !important;
}
#doc-full-content {
  display: block;
  min-height: 100%;
}
#doc-title {
  display: block;
  padding-bottom: 8px;
  border-bottom: 2px solid #e5e7eb;
  margin-bottom: 12px;
}
#doc-title h2 {
  display: block;
  font-size: 1.2rem !important;
  font-weight: 700 !important;
  color: #ffffff !important;
}
"""

with gr.Blocks(css=custom_css) as demo:
    gr.Markdown("# 🔍 노션 트러블슈팅 검색기")
    gr.Markdown("검색어가 없으면 전체 문서 목록, 검색어가 있으면 검색 결과 문서 목록을 표시")

    with gr.Row():
        with gr.Column(scale=4, elem_id="doc-left-panel"):
            with gr.Row():
                search_input = gr.Textbox(
                    label="검색어",
                    placeholder="예: DB 오류, jenkins 설정, alter table...",
                    scale=4,
                )
                search_btn = gr.Button("검색 🔍", scale=1)

                doc_list = gr.Radio(
                    label="📋 문서 목록",
                    choices=[item["title"] for item in build_items("")],
                    value=None,
                    interactive=True,
                )

        with gr.Column(scale=6, elem_id="doc-right-panel"):
          # with gr.Row(): 
          #     doc_title = gr.Markdown(
          #         value="",
          #         elem_id="doc-title"
          #     )
          # with gr.Row(): 
            doc_viewer = gr.Markdown(
                value="문서 제목을 클릭하면 전체 내용을 볼 수 있어요.",
                elem_id="doc-full-content"
            )


    search_btn.click(
        fn=search,
        inputs=search_input,
        # outputs=[doc_list, doc_title, doc_viewer],
        outputs=[doc_list, doc_viewer],
    )

    search_input.submit(
        fn=search,
        inputs=search_input,
        # outputs=[doc_list, doc_title, doc_viewer],
        outputs=[doc_list, doc_viewer],

    )

    doc_list.change(
        fn=show_full_doc,
        inputs=doc_list,
        # outputs=[doc_title, doc_viewer],
          outputs=doc_viewer

    )

    demo.load(
        fn=lambda: search(""),
        # outputs=[doc_list, doc_title, doc_viewer],
        outputs=[doc_list, doc_viewer],

    )

demo.launch()


Running on local URL:  http://127.0.0.1:7895

To create a public link, set `share=True` in `launch()`.
